In [25]:
import wandb
import sys
import matplotlib.pyplot as plt
import scprep
import pandas as pd
# sys.path.append('../hypgs/models')
from omegaconf import OmegaConf
# from train_ae import load_data, make_model
sys.path.append('../../dmae/src/')
from main import load_data
from model import AEDist
import numpy as np
import os
import glob
import scanpy as sc
import torch
from plotly3d.plot import scatter
# os.chdir('/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/hypergraph_scattering/hypgs/models')
import phate

In [2]:
# Initialize wandb (replace 'your_entity' and 'your_project' with your specific details)
wandb.login()
api = wandb.Api()

# Specify your entity, project, and sweep ID
entity = "xingzhis"
project = "dmae"
sweep_id = 'alhw5s0k'

# Fetch the sweep
sweep = api.sweep(f"{entity}/{project}/{sweep_id}")
run_ids = [run.id for run in sweep.runs]

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: xingzhis. Use `wandb login --relogin` to force relogin


In [12]:
run = [run for run in sweep.runs if run.id == run_ids[0]][0]
cfg = OmegaConf.create(run.config)
folder_path = '../../dmae/src/wandb/'
cfg = OmegaConf.create(run.config)
folder_list = glob.glob(f"{folder_path}*{run.id}*")
ckpt_files = glob.glob(f"{folder_list[0]}/files/*.ckpt")
ckpt_path = ckpt_files[0]

data_path = os.path.join(cfg.data.root, cfg.data.name + cfg.data.filetype)
data = np.load(data_path, allow_pickle=True)

model = AEDist.load_from_checkpoint(ckpt_path)
model.eval()

x_all = torch.tensor(data['data'][data['is_train']]).float()
with torch.no_grad():
    x_pred, z_pred = model(x_all)
x_pred = x_pred.detach().cpu().numpy()
z_pred = z_pred.detach().cpu().numpy()
# section = cfg.data.name
# classes = np.load(f'../data/hgsae_fim/fine/{section}_label.npy', allow_pickle=True)
classes = data['colors']
classes = classes[data['is_train']]

/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/pytorch_lightning/utilities/parsing.py:269: UserWarning: Attribute 'activation_fn' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['activation_fn'])`.
  rank_zero_warn(


In [17]:
section_df = pd.read_csv('/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/sea_ad/data/hypergraphae/braak.csv')
section_dict = {section: braak for section, braak in zip(section_df['Section'], section_df['Braak'])}

In [20]:
braak = section_dict[section.strip('.h5ad')]

In [26]:
phate_op = phate.PHATE(n_components=3, random_state=3)
z_phate = phate_op.fit_transform(z_pred)

Calculating PHATE...
  Running PHATE on 3271 observations and 20 variables.
  Calculating graph and diffusion operator...
    Calculating KNN search...
    Calculated KNN search in 0.24 seconds.
    Calculating affinities...
    Calculated affinities in 0.02 seconds.
  Calculated graph and diffusion operator in 0.26 seconds.
  Calculating landmark operator...
    Calculating SVD...
    Calculated SVD in 0.27 seconds.
    Calculating KMeans...
    Calculated KMeans in 4.04 seconds.
  Calculated landmark operator in 5.27 seconds.
  Calculating optimal t...
    Automatically selected t = 38
  Calculated optimal t in 3.10 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 1.30 seconds.
  Calculating metric MDS...


/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/sklearn/manifold/_mds.py:299: FutureWarning:

The default value of `normalized_stress` will change to `'auto'` in version 1.4. To suppress this warning, manually set the value of `normalized_stress`.



  Calculated metric MDS in 85.38 seconds.
Calculated PHATE in 95.35 seconds.


In [28]:
scatter(z_phate, classes, title=braak, s=2)

In [30]:
scatter(phate_op.graph.data_nu[:,:3], classes, title=braak, s=2)